# Dual-Agent Routing framework using ML on A25

**The three things we compare**
- **Single agents**: vision alone, text alone.
- **Router**: a model that picks an agent per cell (Logistic Regression is the baseline; we also try SVM, MLP, Random Forest, Gradient Boosting).
- **Oracle**: an imaginary perfect router; it is the *ceiling* a router can reach, not a method you can deploy.

In [ ]:
from pathlib import Path
import json, re
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

import Levenshtein
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

DATA_ROOT = Path("data/A25/input_images")
VISION_PRED_ROOT = Path("data/outputs/vision_expert_gemma4_run")
TEXT_PRED_ROOT   = Path("data/outputs/text_expert_gemma4_run")
DOMAINS = ["Biology", "CompSci", "ICDAR", "MatSci"]
GT_SUBDIR = "xmls"
VISION_PRED_SUBDIR = "predictions"
TEXT_PRED_SUBDIR = "nougat/predictions"

LEV_THRESHOLD  = 1.0     
RANDOM_STATE   = 27
TEST_SIZE      = 0.25     
VAL_FRACTION   = 0.20  

print("Domains:", DOMAINS)

In [ ]:
MIN_IOU           = 0.9      # matched pairs below this grid-IoU are split back into singletons
MATCH_TEXT_WEIGHT = 0.5      # small lev_sim(text) tiebreaker added to overlapping pairs' score

print(f"Alignment  (MIN_IOU={MIN_IOU}, MATCH_TEXT_WEIGHT={MATCH_TEXT_WEIGHT})")


## Step 1 — Helper files

In [ ]:

def normalize_text(text):

    if text is None:
        return ""
    
    text = str(text).lower()
    text = text.replace("\\times", "x")
    text = text.replace(chr(0x2212), "-").replace(chr(0x2013), "-").replace(chr(0x2014), "-")
    text = re.sub(r"\$\^\{(\d+)\}\$", r"\1", text)
    text = text.replace("$", "")
    return re.sub(r"\s+", " ", text).strip()

def lev_sim(a, b):
    a, b = normalize_text(a), normalize_text(b)

    if a == "" and b == "":
        return 1.0
    
    return 1.0 - Levenshtein.distance(a, b) / max(len(a), len(b), 1)

def cell_key(cell):
    return (int(cell["sr"]), int(cell["er"]), int(cell["sc"]), int(cell["ec"]))

def parse_gt_xml(path):
    if path is None or not Path(path).exists():
        return []
    
    try:
        root = ET.parse(path).getroot()
    except Exception:
        return []
    
    out = []
    for c in root.findall("cell"):

        sr = c.get("start_row")
        sc = c.get("start_col")

        if sr is None or sc is None:
            continue

        er = c.get("end_row", sr)
        ec = c.get("end_col", sc)

        text_el = c.find("text")
        text = (text_el.text or "") if text_el is not None else ""

        out.append({
            "sr": int(sr), 
            "er": int(er),
            "sc": int(sc), 
            "ec": int(ec),
            "text": normalize_text(text)
        })
    return out

def parse_pred_json(path):
    if path is None or not Path(path).exists():
        return []
    
    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return []
    
    cells = data.get("cells", data) if isinstance(data, dict) else data
    if not isinstance(cells, list):
        return []
    
    out = []
    for c in cells:
        if not isinstance(c, dict):
            continue

        sr, sc = c.get("sr", c.get("start_row")), c.get("sc", c.get("start_col"))
        if sr is None or sc is None:
            continue

        er, ec = c.get("er", c.get("end_row", sr)), c.get("ec", c.get("end_col", sc))
        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec),
            "text": normalize_text(c.get("text", ""))
        })
    return out

def pred_lookup(pred_cells):
    out = {}
    for c in pred_cells:
        out.setdefault(cell_key(c), c)

    return out

def table_avg(df, col):
    return float(df.groupby("table_id")[col].mean().mean()) if len(df) else 0.0

def strict_correct(gt_cell, pred_cell, threshold=LEV_THRESHOLD):
    
    if pred_cell is None:
        return {"sim": 0.0, "correct": False}
    
    sim = lev_sim(gt_cell["text"], pred_cell["text"])

    return {
        "sim": sim, 
        "correct": bool(sim >= threshold)
    }

## Step 2 — Find every table and choose the evaluation protocol

A25 is organized by domain (Biology, CompSci, ICDAR, MatSci) with no pre-defined train/test split, so we pool all tables across domains and split them ourselves by table ID.=

In [ ]:

def discover_tables(domains):
    rows = []
    for domain in domains:

        gt_dir  = DATA_ROOT / domain / GT_SUBDIR
        vis_dir = VISION_PRED_ROOT / domain / VISION_PRED_SUBDIR
        txt_dir = TEXT_PRED_ROOT / domain / TEXT_PRED_SUBDIR

        if not gt_dir.exists():
            print(f"[warn] no ground-truth folder for domain '{domain}': {gt_dir}")
            continue

        gt  = {p.stem: p for p in sorted(gt_dir.glob("*.xml"))}
        vis = {p.stem: p for p in vis_dir.glob("*.json")} if vis_dir.exists() else {}
        txt = {p.stem: p for p in txt_dir.glob("*.json")} if txt_dir.exists() else {}

        for s, g in gt.items():
            rows.append({
                "domain": domain, 
                "stem": s, 
                "table_id": f"{domain}::{s}",
                "gt_file": g,
                "vision_file": vis.get(s),
                "text_file": txt.get(s),
                "vision_missing_file": s not in vis, 
                "text_missing_file": s not in txt,
            })
    return pd.DataFrame(rows)

all_tables_raw = discover_tables(DOMAINS)

def filter_predicted(df):
    before = len(df)
    out = df[~df["vision_missing_file"] & ~df["text_missing_file"]].copy().reset_index(drop=True)
    dropped = before - len(out)

    if dropped:
        print(f"  [filter] dropped {dropped} tables with missing predictions, kept {len(out)}")
        
    return out

all_tables = filter_predicted(all_tables_raw)

print(f"Total tables (with both predictions): {len(all_tables)}")
print(all_tables.groupby("domain").size().rename("tables").to_frame())

## Step 3 — Turn each cell into one labeled row

In [ ]:

import unicodedata

MARKUP_RE  = re.compile(r'\\[a-zA-Z]+|[_^]\{')
NUMERIC_RE = re.compile(r'^[\d\s\.\-\+±×xX\^\(\)\[\]\/%,;:<>=]+$')

def has_symbol(t):
    t = normalize_text(t)

    if MARKUP_RE.search(t):
        return True
    
    return any(ord(ch) > 127 and (unicodedata.category(ch)[0] == "S" or "GREEK" in unicodedata.name(ch, "")) for ch in t)

def is_numeric(t): 
    t = normalize_text(t)
    return bool(t) and bool(NUMERIC_RE.match(t)) and bool(re.search(r'\d', t))

def safe_text(c):  
    return "" if c is None else normalize_text(c.get("text", ""))

def span_rows(c):  
    return 0 if c is None else int(c["er"]) - int(c["sr"]) + 1

def span_cols(c):  
    return 0 if c is None else int(c["ec"]) - int(c["sc"]) + 1

def router_features(v, t):
    vt, tt = safe_text(v), safe_text(t)
    vr, vc, tr, tc = span_rows(v), span_cols(v), span_rows(t), span_cols(t)
    return {
        "vision_missing": v is None, 
        "text_missing": t is None,
        "one_missing": (v is None) != (t is None),
        "vision_len": len(vt), 
        "text_len": len(tt), 
        "len_diff": abs(len(vt) - len(tt)),
        "vision_empty": vt == "", 
        "text_empty": tt == "", 
        "empty_mismatch": (vt == "") != (tt == ""),
        "same_text": vt == tt, 
        "agent_similarity": lev_sim(vt, tt),
        "vision_numeric": is_numeric(vt), 
        "text_numeric": is_numeric(tt),
        "numeric_mismatch": is_numeric(vt) != is_numeric(tt),
        "vision_symbol": has_symbol(vt), 
        "text_symbol": has_symbol(tt),
        "symbol_mismatch": has_symbol(vt) != has_symbol(tt),
        "row_span_diff": abs(vr - tr), 
        "col_span_diff": abs(vc - tc),
        "vision_merged": (vr > 1) or (vc > 1),
        "text_merged": (tr > 1) or (tc > 1),
    }

FEATURES = list(router_features(None, None).keys())
print(f"Router uses {len(FEATURES)} prediction-only features.")

def build_cell_dataset(table_df):
    rows = []
    for _, tab in table_df.iterrows():

        gt = parse_gt_xml(tab["gt_file"])
        vl = pred_lookup(parse_pred_json(tab["vision_file"]))
        tl = pred_lookup(parse_pred_json(tab["text_file"]))

        for gi, g in enumerate(gt):

            k = cell_key(g)
            vcell, tcell = vl.get(k), tl.get(k)
            v, t = strict_correct(g, vcell), strict_correct(g, tcell)

            rows.append({
                "domain": tab["domain"], 
                "stem": tab["stem"], 
                "table_id": tab["table_id"],
                "gt_cell_idx": gi, 
                "gt_text": g["text"],
                "vision_text": safe_text(vcell), 
                "text_text": safe_text(tcell),
                "vision_correct": bool(v["correct"]), 
                "text_correct": bool(t["correct"]),
                "oracle_correct": bool(v["correct"] or t["correct"]),
                "one_agent_correct": bool(v["correct"] != t["correct"]),
                "choose_vision": int(v["correct"] and not t["correct"]),
                **router_features(vcell, tcell),
            })
    return pd.DataFrame(rows)

## Step 3b — Align the two agents to each other


In [ ]:
from scipy.optimize import linear_sum_assignment

def agent_iou(v, t):
    """IoU of two cells' grid footprints (rows sr..er x cols sc..ec)"""
    if v is None or t is None:
        return 0.0
    
    inter_r = min(v["er"], t["er"]) - max(v["sr"], t["sr"]) + 1
    inter_c = min(v["ec"], t["ec"]) - max(v["sc"], t["sc"]) + 1

    if inter_r <= 0 or inter_c <= 0:
        return 0.0
    
    inter  = inter_r * inter_c
    area_v = (v["er"] - v["sr"] + 1) * (v["ec"] - v["sc"] + 1)
    area_t = (t["er"] - t["sr"] + 1) * (t["ec"] - t["sc"] + 1)

    return inter / (area_v + area_t - inter)

def match_agents(vision_cells, text_cells, min_iou=None, text_weight=None):
    min_iou = MIN_IOU if min_iou is None else min_iou
    text_weight = MATCH_TEXT_WEIGHT if text_weight is None else text_weight

    if not vision_cells or not text_cells:        
        return [(v, None) for v in vision_cells] + [(None, t) for t in text_cells]
    
    iou = np.array([[agent_iou(v, t) for t in text_cells] for v in vision_cells])
    S = iou.copy()

    if text_weight:
        for i, v in enumerate(vision_cells):     
            for j, t in enumerate(text_cells):
                if iou[i, j] > 0:
                    S[i, j] += text_weight * lev_sim(v["text"], t["text"])

    vi, tj = linear_sum_assignment(-S)            
    units, used_v, used_t = [], set(), set()

    for i, j in zip(vi, tj):
        if iou[i, j] >= min_iou:
            units.append((vision_cells[i], text_cells[j]))
            used_v.add(i); used_t.add(j)

    units += [(vision_cells[i], None) for i in range(len(vision_cells)) if i not in used_v]
    units += [(None, text_cells[j]) for j in range(len(text_cells)) if j not in used_t]

    return units


def gt_lookup(gt_cells):
    out = {}
    for g in gt_cells:
        out.setdefault(cell_key(g), g)
    return out

def cell_correct_vs_gt(pred_cell, gt_by_key, threshold=LEV_THRESHOLD):

    if pred_cell is None:
        return False
    
    g = gt_by_key.get(cell_key(pred_cell))

    return g is not None and lev_sim(g["text"], pred_cell["text"]) >= threshold

def build_unit_dataset(table_df):
    rows = []
    for _, tab in table_df.iterrows():

        gt_by_key = gt_lookup(parse_gt_xml(tab["gt_file"]))
        units = match_agents(parse_pred_json(tab["vision_file"]), parse_pred_json(tab["text_file"]))

        for ui, (vcell, tcell) in enumerate(units):

            vc, tc = cell_correct_vs_gt(vcell, gt_by_key), cell_correct_vs_gt(tcell, gt_by_key)

            rows.append({
                "domain": tab["domain"], 
                "stem": tab["stem"],
                "table_id": tab["table_id"],
                "unit_idx": ui, "vision_cell": vcell, 
                "text_cell": tcell,
                "vision_text": safe_text(vcell),
                "text_text": safe_text(tcell),
                "vision_correct": bool(vc), 
                "text_correct": bool(tc),
                "oracle_correct": bool(vc or tc),
                "one_agent_correct": bool(vc != tc),
                "choose_vision": int(vc and not tc),
                **router_features(vcell, tcell),
            })
    return pd.DataFrame(rows)

## Step 4 — Train / validation / test split (by table)

The split is always by table.

In [ ]:
def split_by_table(df, test_size, val_size, seed):

    g1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trv_i, te_i = next(g1.split(df, groups=df["table_id"]))
    trv, te = df.iloc[trv_i].copy(), df.iloc[te_i].copy()

    g2 = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    tr_i, va_i = next(g2.split(trv, groups=trv["table_id"]))

    return trv.iloc[tr_i].copy(), trv.iloc[va_i].copy(), te


df_all = build_unit_dataset(all_tables)
df_train, df_val, df_test = split_by_table(df_all, test_size=TEST_SIZE, val_size=VAL_FRACTION, seed=RANDOM_STATE)

print(pd.DataFrame({
    "tables": [d["table_id"].nunique() for d in (df_train, df_val, df_test)],
    "cells":  [len(d) for d in (df_train, df_val, df_test)],
}, index=["train", "val", "test"]))

## Comparison 1 — Single agents vs the oracle

In [ ]:



tables_by_id = all_tables.set_index("table_id")
test_ids  = sorted(df_test["table_id"].unique())

def gt_anchored_baselines(table_ids):
    rows = []

    for tid in table_ids:
        tab = tables_by_id.loc[tid]

        gt = parse_gt_xml(tab["gt_file"])
        vl = pred_lookup(parse_pred_json(tab["vision_file"]))
        tl = pred_lookup(parse_pred_json(tab["text_file"]))

        for g in gt:
            k = cell_key(g)
            v, t = strict_correct(g, vl.get(k)), strict_correct(g, tl.get(k))

            rows.append({
                "table_id": tid,
                "vision_correct": bool(v["correct"]),
                "text_correct": bool(t["correct"]),
                "oracle_correct": bool(v["correct"] or t["correct"])
            })
    return pd.DataFrame(rows)

def score_router_gt_anchored(routed_df, pick_col):
    rows = []

    for tid, grp in routed_df.groupby("table_id"):

        chosen = [r["vision_cell"] if r[pick_col] else r["text_cell"] for _, r in grp.iterrows()]
        rl = pred_lookup([c for c in chosen if c is not None])

        for g in parse_gt_xml(tables_by_id.loc[tid]["gt_file"]):

            s = strict_correct(g, rl.get(cell_key(g)))
            rows.append({"table_id": tid, "router_correct": bool(s["correct"])})

    return pd.DataFrame(rows)

gt_base = gt_anchored_baselines(test_ids)
vis_acc = table_avg(gt_base, "vision_correct")
txt_acc = table_avg(gt_base, "text_correct")
best_single = max(vis_acc, txt_acc)
oracle = table_avg(gt_base, "oracle_correct")
oracle_gap  = oracle - best_single

print(f"Vision: {vis_acc*100:.2f}%  |  Text: {txt_acc*100:.2f}%  |  "
      f"Best single: {best_single*100:.2f}%  |  Oracle: {oracle*100:.2f}%  |  "
      f"Oracle gap: {oracle_gap*100:.2f} pp")

## Comparison 2 — Logistic Regression router

The simplest learned router. It is trained only on cells where exactly one agent is
right (the only cells that teach a preference).

In [ ]:
def add_router(df, model, threshold=0.5, prefix="r"):
    out = df.copy()

    X = out[FEATURES].astype(float)
    p = model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else 1.0 / (1.0 + np.exp(-model.decision_function(X)))

    out[f"{prefix}_p"] = p
    out[f"{prefix}_pick_vision"] = p >= threshold
    out[f"{prefix}_correct"] = np.where(out[f"{prefix}_pick_vision"], out["vision_correct"], out["text_correct"]).astype(bool)

    return out

def tune_threshold(model, val_df, prefix="t"):

    best_thr, best_acc = 0.5, -1.0
    for thr in np.round(np.arange(0.20, 0.81, 0.05), 2):

        acc = table_avg(add_router(val_df, model, threshold=thr, prefix=prefix), f"{prefix}_correct")

        if acc > best_acc:
            best_acc, best_thr = acc, float(thr)

    return best_thr, best_acc

train_router = df_train[df_train["one_agent_correct"]].copy()

lr = Pipeline([
    ("scaler", StandardScaler()), 
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
])
lr.fit(train_router[FEATURES].astype(float), train_router["choose_vision"].astype(int))

lr_thr, _ = tune_threshold(lr, df_val, prefix="lr")
lr_routed = add_router(df_test, lr, threshold=lr_thr, prefix="lr")
lr_acc = table_avg(score_router_gt_anchored(lr_routed, "lr_pick_vision"), "router_correct")
lr_gain = lr_acc - best_single

table1 = pd.DataFrame([{
    "set": "test",
    "averaging": "table-average",
    "tables": int(len(test_ids)),
    "cells": int(len(gt_base)),
    "Vision acc %": round(vis_acc * 100, 2),
    "Text acc %":round(txt_acc * 100, 2),
    "Best single %": round(best_single * 100, 2),
    "LR router acc %": round(lr_acc * 100, 2),
    "Oracle acc %": round(oracle * 100, 2),
    "LR gain pp":round(lr_gain * 100, 2),
    "Oracle gap pp": round(oracle_gap * 100, 2),
    "Gap closed %": round(lr_gain / oracle_gap * 100, 2) if oracle_gap > 0 else 0.0,
}])

display(table1)

## Comparison 3 — Many router models

Trying on different model families: Logistic Regression, SVM, MLP, Random Forest, Gradient Boosting. Each is tuned **only on validation**.

In [ ]:

ROUTER_SPECS = {
    "LR router": {
        "model": Pipeline([("scaler", StandardScaler()),
                           ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))]),
        "grid": [{"clf__C": c} for c in [0.1, 1.0, 10.0]],
    },
    "SVM": {
        "model": Pipeline([("scaler", StandardScaler()),
                           ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE))]),
        "grid": [{"clf__C": c, "clf__gamma": g} for c in [1.0, 10.0] for g in ["scale", 0.1]],
    },
    "Small MLP": {
        "model": Pipeline([("scaler", StandardScaler()),
                           ("clf", MLPClassifier(max_iter=800, early_stopping=True, random_state=RANDOM_STATE))]),
        "grid": [{"clf__hidden_layer_sizes": h, "clf__alpha": a} for h in [(32,), (64, 32)] for a in [1e-4, 1e-2]],
    },
    "Random Forest": {
        "model": Pipeline([("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))]),
        "grid": [{"clf__n_estimators": 300, "clf__max_depth": d} for d in [None, 6, 12]],
    },
    "Gradient Boosting": {
        "model": Pipeline([("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))]),
        "grid": [{"clf__n_estimators": 100, "clf__learning_rate": lr_, "clf__max_depth": d}
                 for lr_ in [0.05, 0.1] for d in [2, 3]],
    },
}

def tune_and_fit(name, spec, train_router_df, val_df):

    if train_router_df["choose_vision"].nunique() < 2:
        raise ValueError(f"{name}: need both vision-win and text-win training cells.")
    
    X = train_router_df[FEATURES].astype(float)
    y = train_router_df["choose_vision"].astype(int)

    best = None
    for params in spec["grid"]:

        m = clone(spec["model"]).set_params(**params)
        m.fit(X, y)
        thr, val_acc = tune_threshold(m, val_df, prefix="tmp")

        if best is None or val_acc > best["val_acc"]:
            best = {"name": name, "model": m, "threshold": thr, "val_acc": val_acc, "params": params}

    return best

fitted, table2_rows = {}, []
for name, spec in ROUTER_SPECS.items():

    b = tune_and_fit(name, spec, train_router, df_val)
    fitted[name] = b

    routed = add_router(df_test, b["model"], threshold=b["threshold"], prefix="r")

    r_acc = table_avg(score_router_gt_anchored(routed, "r_pick_vision"), "router_correct")
    r_gain = r_acc - best_single
    
    table2_rows.append({
        "set": "test",
        "method": name,
        "averaging": "table-average",
        "tables": int(len(test_ids)),
        "cells": int(len(gt_base)),
        "Vision acc %": round(vis_acc * 100, 2),
        "Text acc %": round(txt_acc * 100, 2),
        "Best single %":round(best_single * 100, 2),
        "Router acc %": round(r_acc * 100, 2),
        "Oracle acc %": round(oracle * 100, 2),
        "Gain over best single pp": round(r_gain * 100, 2),
        "Oracle gap pp":round(oracle_gap * 100, 2),
        "Gap closed %": round(r_gain / oracle_gap * 100, 2) if oracle_gap > 0 else 0.0,
    })

table2 = pd.DataFrame(table2_rows).sort_values("Gain over best single pp", ascending=False)
display(table2)